# ⚡ XGBoost - Tuned Model & GPU Acceleration

- **Thành viên:** B (Cao Anh Minh)
- **Mã thí nghiệm:** `xgb_b_v1`
- **Kết quả:** CV AUC = **0.94697**, Public LB = **0.94759**
- **Đặc điểm:** Huấn luyện trên GPU với histogram tree method (`tree_method="hist"`), độ sâu cây tối ưu `max_depth=4`, `min_child_weight=20`, `colsample_bytree=0.8`.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

ROOT = Path.cwd().parent if Path.cwd().name == "B" else Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "B" / "outputs"
SUB_DIR = ROOT / "submissions"
EXP_ID = "xgb_b_v1"
SEED = 42
N_SPLITS = 5


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
folds_df = pd.read_csv(DATA_DIR / "folds.csv")
sub_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_df["fold"] = folds_df["fold"]

cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
bool_cols = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]

for c in bool_cols:
    train_df[c] = train_df[c].astype(int)
    test_df[c] = test_df[c].astype(int)

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_df[cat_cols] = oe.fit_transform(train_df[cat_cols])
test_df[cat_cols] = oe.transform(test_df[cat_cols])

features = [c for c in train_df.columns if c not in ["id", "is_fraud", "fold"]]
print("Features count:", len(features))


In [ ]:
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.04,
    "max_depth": 4,
    "min_child_weight": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "n_estimators": 2500,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": SEED,
    "early_stopping_rounds": 50
}

oof_preds = np.zeros(len(train_df), dtype=np.float64)
test_preds = np.zeros(len(test_df), dtype=np.float64)
fold_aucs = []

for fold in range(N_SPLITS):
    tr_mask = train_df["fold"] != fold
    va_mask = train_df["fold"] == fold
    X_train, y_train = train_df.loc[tr_mask, features], train_df.loc[tr_mask, "is_fraud"]
    X_val, y_val = train_df.loc[va_mask, features], train_df.loc[va_mask, "is_fraud"]

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[va_mask] = val_pred
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_aucs.append(fold_auc)
    test_preds += model.predict_proba(test_df[features])[:, 1] / N_SPLITS
    print("Fold", fold+1, "AUC:", round(fold_auc, 5), "Best iter:", model.best_iteration)

overall_auc = roc_auc_score(train_df["is_fraud"], oof_preds)
print("Mean Fold AUC:", round(np.mean(fold_aucs), 5), "+/-", round(np.std(fold_aucs), 5))
print("Overall OOF AUC:", round(overall_auc, 5))


In [ ]:
# Lưu kết quả
np.save(OUTPUT_DIR / f"{EXP_ID}_oof.npy", oof_preds)
np.save(OUTPUT_DIR / f"{EXP_ID}_test.npy", test_preds)
sub_df["is_fraud"] = test_preds
sub_df.to_csv(SUB_DIR / f"{EXP_ID}.csv", index=False)
print("Saved outputs and submission successfully!")
